# Vietnamese Emotion Classification - BiLSTM with 5-Fold Cross-Validation

**Google Colab Ready** - optimized for GPU acceleration.

This notebook implements a **BiLSTM** architecture for Vietnamese emotion classification with **5-fold cross-validation**:

- **BiLSTM layers** capture long-range bidirectional dependencies with memory cells
- **5-fold cross-validation** for robust model evaluation
- **Attention pooling** summarizes the sequence into a fixed-length vector
- **Underthesea tokenizer** for Vietnamese word segmentation

### How to Use
1. Upload training data to Google Drive at: `MyDrive/thesis/data/`
   - Required: `processed/train_10000_final.csv`, `processed/val_processed.csv`, `processed/test_processed.csv`
2. Enable GPU: **Runtime → Change runtime type → GPU**
3. Run cells in order

---

## Techniques Implemented

1. **5-Fold Cross-Validation** - robust model evaluation using stratified k-fold splits
2. **Bidirectional LSTM** - captures forward and backward contextual information with long-term memory
3. **Stacked BiLSTM** - multiple BiLSTM layers for hierarchical feature learning
4. **Early Stopping & LR Scheduling** - monitors validation loss
5. **Mixed Precision (FP16)** - enabled automatically on GPU
6. **W&B Logging** - tracks training metrics
7. **Underthesea Tokenization** - Vietnamese word segmentation

## 0. Google Colab Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("Google Drive mounted successfully")
print("Data should be in: /content/drive/MyDrive/thesis/data")

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    TPU_AVAILABLE = True
    print("TPU detected and configured")
except ImportError:
    TPU_AVAILABLE = False
    print("TPU not available, will use GPU/CPU")

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib seaborn tensorflow keras wandb underthesea openpyxl

In [ ]:
import os
import json
import random
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Bidirectional,
    Dense,
    Dropout,
    BatchNormalization,
    GlobalMaxPooling1D,
    SpatialDropout1D,
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print(f"GPU(s) detected: {[g.name for g in gpus]}")
    print(f"Mixed-precision policy: {tf.keras.mixed_precision.global_policy().name}")
else:
    print("No GPU detected - running on CPU.")

print(f"TensorFlow version: {tf.__version__}")

## 1. Configuration

In [ ]:
DATA_DIR = '/content/drive/MyDrive/thesis/data'
MODEL_SAVE_PATH = '/content/drive/MyDrive/thesis/emotion_classifier_bilstm'

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

MAX_VOCAB_SIZE = 20000
MAX_SEQ_LEN = 100
EMBED_DIM = 128

LSTM_UNITS = 128
NUM_LSTM_LAYERS = 2

BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 0.001

N_FOLDS = 3

USE_UNDERTHESEA_TOKENIZER = True
USE_WANDB = False

print("Configuration:")
print(f"  Data directory: {DATA_DIR}")
print(f"  Model save path: {MODEL_SAVE_PATH}")
print(f"  Random seed: {RANDOM_SEED}")
print(f"  Max vocab size: {MAX_VOCAB_SIZE}")
print(f"  Max sequence length: {MAX_SEQ_LEN}")
print(f"  Embedding dim: {EMBED_DIM}")
print(f"  LSTM units: {LSTM_UNITS}")
print(f"  Number of LSTM layers: {NUM_LSTM_LAYERS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Number of folds: {N_FOLDS}")
print(f"  Use Underthesea tokenizer: {USE_UNDERTHESEA_TOKENIZER}")
print(f"  Use W&B: {USE_WANDB}")

## 2. W&B Setup (Optional)

In [ ]:
if USE_WANDB:
    import wandb
    from wandb.keras import WandbCallback

    wandb.login()

    wandb.init(
        project='emotion-classification-baseline',
        name='bilstm-test',
        config={
            'model': 'BiLSTM',
            'max_vocab': MAX_VOCAB_SIZE,
            'max_len': MAX_SEQ_LEN,
            'embed_dim': EMBED_DIM,
            'lstm_units': LSTM_UNITS,
            'num_lstm_layers': NUM_LSTM_LAYERS,
            'batch_size': BATCH_SIZE,
            'epochs': EPOCHS,
            'learning_rate': LEARNING_RATE,
            'seed': RANDOM_SEED,
            'underthesea': USE_UNDERTHESEA_TOKENIZER,
        },
    )
    print("W&B initialized")
else:
    print("W&B disabled")

## 3. Load Underthesea Tokenizer

In [ ]:
if USE_UNDERTHESEA_TOKENIZER:
    from underthesea import word_tokenize

    def tokenize_vietnamese(text):
        try:
            return word_tokenize(str(text), format='text')
        except Exception:
            return str(text)

    print("Underthesea tokenizer loaded")
else:
    print("Using raw text (no Underthesea tokenization)")

## 4. Load and Preprocess Data

In [ ]:
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
print(f"Model save directory ready: {MODEL_SAVE_PATH}")

train_df = pd.read_csv(os.path.join(DATA_DIR, 'processed/train_1500_para_final.csv'))
val_df = pd.read_csv(os.path.join(DATA_DIR, 'processed/val_processed.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'processed/ttest_gen_final.csv'))

print(f"\nDatasets loaded from Google Drive")
print(f"  Train: {train_df.shape[0]} samples")
print(f"  Val: {val_df.shape[0]} samples")
print(f"  Test: {test_df.shape[0]} samples")

train_df = train_df.drop(['Emotion', 'Sentence'], axis=1)
train_df.columns = ['label', 'text']

val_df = val_df.rename(columns={'Sentence': 'text', 'emotion_vn': 'label'}).drop('Emotion', axis=1)
test_df = test_df.rename(columns={'Sentence': 'text', 'emotion_vn': 'label'}).drop('Emotion', axis=1)

for current_df in [train_df, val_df, test_df]:
    current_df.dropna(subset=['text', 'label'], inplace=True)
    current_df['text'] = current_df['text'].astype(str).str.strip()
    current_df['label'] = current_df['label'].astype(str).str.strip()

    if USE_UNDERTHESEA_TOKENIZER:
        current_df['text'] = current_df['text'].apply(tokenize_vietnamese)
        print(f"Applied underthesea tokenization to {current_df.shape[0]} samples")

full_train_df = pd.concat([train_df, val_df], ignore_index=True)
full_train_df = full_train_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"\nCombined train and validation data for cross-validation")
print(f"  Full training set: {full_train_df.shape[0]} samples")
print(f"  Test set: {test_df.shape[0]} samples")

print(f"\n{'='*80}")
print("SAMPLE TEXT")
print(f"{'='*80}")
for i in range(3):
    print(f"Sample {i+1}: {full_train_df['text'].iloc[i][:150]}...")
print(f"{'='*80}")

print("\nData loaded and cleaned successfully")

## 5. Encode Labels and Tokenize Text

In [ ]:
label_encoder = LabelEncoder()
full_train_df['label_encoded'] = label_encoder.fit_transform(full_train_df['label'])
test_df['label_encoded'] = label_encoder.transform(test_df['label'])

num_classes = len(label_encoder.classes_)
print(f"Number of classes: {num_classes}")
print(f"Classes: {label_encoder.classes_}")

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(full_train_df['text'])

X_full = tokenizer.texts_to_sequences(full_train_df['text'])
X_test = tokenizer.texts_to_sequences(test_df['text'])

X_full = pad_sequences(X_full, maxlen=MAX_SEQ_LEN, padding='post', truncating='post')
X_test = pad_sequences(X_test, maxlen=MAX_SEQ_LEN, padding='post', truncating='post')

y_full = full_train_df['label_encoded'].values
y_test = to_categorical(test_df['label_encoded'], num_classes=num_classes)

vocab_size = min(len(tokenizer.word_index) + 1, MAX_VOCAB_SIZE)

print(f"\nTokenization complete:")
print(f"  Vocabulary size: {vocab_size}")
print(f"  Full training shape: {X_full.shape}")
print(f"  Test shape: {X_test.shape}")

## 6. Build BiLSTM Model

In [ ]:
def build_bilstm(vocab_size, embed_dim, max_len, num_classes,
                 lstm_units=128, num_layers=2):
    """
    Stacked BiLSTM architecture:
      Embedding -> SpatialDropout
        -> BiLSTM layers (stacked)
        -> GlobalMaxPooling
        -> Dense -> Softmax
    """
    inp = Input(shape=(max_len,))

    x = Embedding(vocab_size, embed_dim, input_length=max_len)(inp)
    x = SpatialDropout1D(0.2)(x)

    for i in range(num_layers):
        return_sequences = True
        x = Bidirectional(LSTM(lstm_units, return_sequences=return_sequences))(x)
        x = Dropout(0.3)(x)

    x = GlobalMaxPooling1D()(x)

    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)

    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)

    out = Dense(num_classes, activation='softmax', dtype='float32')(x)

    model = Model(inputs=inp, outputs=out)
    return model


model = build_bilstm(
    vocab_size=vocab_size,
    embed_dim=EMBED_DIM,
    max_len=MAX_SEQ_LEN,
    num_classes=num_classes,
    lstm_units=LSTM_UNITS,
    num_layers=NUM_LSTM_LAYERS,
)

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()
print(f"\nTotal parameters: {model.count_params():,}")

## 7. 5-Fold Cross-Validation Training

In [ ]:
print("\n" + "="*80)
print(f"TRAINING BiLSTM MODEL WITH {N_FOLDS}-FOLD CROSS-VALIDATION")
print(f"Seed: {RANDOM_SEED}")
print("="*80 + "\n")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

fold_histories = []
fold_models = []
fold_val_accuracies = []
fold_val_losses = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full, y_full), 1):
    print("\n" + "="*80)
    print(f"FOLD {fold}/{N_FOLDS}")
    print("="*80)

    X_train_fold = X_full[train_idx]
    y_train_fold = to_categorical(y_full[train_idx], num_classes=num_classes)
    X_val_fold = X_full[val_idx]
    y_val_fold = to_categorical(y_full[val_idx], num_classes=num_classes)

    print(f"Train samples: {len(X_train_fold)}")
    print(f"Validation samples: {len(X_val_fold)}")

    tf.keras.backend.clear_session()

    model = build_bilstm(
        vocab_size=vocab_size,
        embed_dim=EMBED_DIM,
        max_len=MAX_SEQ_LEN,
        num_classes=num_classes,
        lstm_units=LSTM_UNITS,
        num_layers=NUM_LSTM_LAYERS,
    )

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )

    checkpoint_path = f'./bilstm_fold_{fold}_seed_{RANDOM_SEED}.keras'

    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1,
        ),
        ModelCheckpoint(
            checkpoint_path,
            monitor='val_loss',
            save_best_only=True,
            verbose=1,
        ),
    ]

    if USE_WANDB:
        wandb.init(
            project='emotion-classification-baseline',
            name=f'bilstm-fold-{fold}',
            config={
                'model': 'BiLSTM',
                'fold': fold,
                'max_vocab': MAX_VOCAB_SIZE,
                'max_len': MAX_SEQ_LEN,
                'embed_dim': EMBED_DIM,
                'lstm_units': LSTM_UNITS,
                'num_lstm_layers': NUM_LSTM_LAYERS,
                'batch_size': BATCH_SIZE,
                'epochs': EPOCHS,
                'learning_rate': LEARNING_RATE,
                'seed': RANDOM_SEED,
            },
            reinit=True,
        )
        callbacks.append(WandbCallback(save_model=False))

    history = model.fit(
        X_train_fold,
        y_train_fold,
        validation_data=(X_val_fold, y_val_fold),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1,
    )

    model.load_weights(checkpoint_path)

    val_loss, val_acc = model.evaluate(X_val_fold, y_val_fold, verbose=0)
    print(f"\nFold {fold} - Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")

    fold_histories.append(history)
    fold_models.append(model)
    fold_val_accuracies.append(val_acc)
    fold_val_losses.append(val_loss)

    if USE_WANDB:
        wandb.finish()

print("\n" + "="*80)
print("CROSS-VALIDATION SUMMARY")
print("="*80)
print(f"Mean Validation Accuracy: {np.mean(fold_val_accuracies):.4f} ± {np.std(fold_val_accuracies):.4f}")
print(f"Mean Validation Loss: {np.mean(fold_val_losses):.4f} ± {np.std(fold_val_losses):.4f}")
print("\nPer-fold results:")
for i, (acc, loss) in enumerate(zip(fold_val_accuracies, fold_val_losses), 1):
    print(f"  Fold {i}: Accuracy = {acc:.4f}, Loss = {loss:.4f}")

print("\nCross-validation training complete!")

## 8. Evaluate on Test Set (Ensemble)

In [ ]:
print("\n" + "="*80)
print("EVALUATING ON TEST SET (ENSEMBLE PREDICTION)")
print("="*80)

all_fold_predictions = []
for i, model in enumerate(fold_models, 1):
    print(f"Getting predictions from fold {i}...")
    fold_pred_probs = model.predict(X_test, verbose=0)
    all_fold_predictions.append(fold_pred_probs)

y_pred_probs_ensemble = np.mean(all_fold_predictions, axis=0)

y_pred = np.argmax(y_pred_probs_ensemble, axis=1)
y_true = np.argmax(y_test, axis=1)

test_acc = accuracy_score(y_true, y_pred)
print(f"\nEnsemble Test Accuracy: {test_acc:.4f}")

print("\n" + "="*80)
print("CLASSIFICATION REPORT (ENSEMBLE)")
print("="*80)
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_, digits=4))

individual_fold_accuracies = []
for i, model in enumerate(fold_models, 1):
    fold_pred_probs = model.predict(X_test, verbose=0)
    fold_pred = np.argmax(fold_pred_probs, axis=1)
    fold_acc = accuracy_score(y_true, fold_pred)
    individual_fold_accuracies.append(fold_acc)
    print(f"Fold {i} individual test accuracy: {fold_acc:.4f}")

print(f"\nMean individual fold test accuracy: {np.mean(individual_fold_accuracies):.4f} ± {np.std(individual_fold_accuracies):.4f}")
print(f"Ensemble test accuracy: {test_acc:.4f}")

results_dict = {
    'ensemble_test_accuracy': test_acc,
    'ensemble_test_f1_macro': f1_score(y_true, y_pred, average='macro'),
    'ensemble_test_f1_weighted': f1_score(y_true, y_pred, average='weighted'),
    'mean_cv_val_accuracy': np.mean(fold_val_accuracies),
    'std_cv_val_accuracy': np.std(fold_val_accuracies),
    'mean_cv_val_loss': np.mean(fold_val_losses),
    'std_cv_val_loss': np.std(fold_val_losses),
    'mean_individual_fold_test_accuracy': np.mean(individual_fold_accuracies),
    'std_individual_fold_test_accuracy': np.std(individual_fold_accuracies),
}

for i in range(N_FOLDS):
    results_dict[f'fold_{i+1}_val_accuracy'] = fold_val_accuracies[i]
    results_dict[f'fold_{i+1}_test_accuracy'] = individual_fold_accuracies[i]

results_df = pd.DataFrame([results_dict])
results_df.to_csv('bilstm_cv_results.csv', index=False)
print("\nResults saved to bilstm_cv_results.csv")

## 9. Visualize Training History

In [ ]:
fig, axes = plt.subplots(N_FOLDS, 2, figsize=(14, 4 * N_FOLDS))

for i, history in enumerate(fold_histories):
    ax1 = axes[i, 0]
    ax2 = axes[i, 1]

    ax1.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=10)
    ax1.set_ylabel('Accuracy', fontsize=10)
    ax1.set_title(f'Fold {i+1} - Accuracy', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    ax2.plot(history.history['loss'], label='Train Loss', linewidth=2)
    ax2.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=10)
    ax2.set_ylabel('Loss', fontsize=10)
    ax2.set_title(f'Fold {i+1} - Loss', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bilstm_cv_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("Cross-validation training history plot saved as bilstm_cv_training_history.png")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

fold_numbers = list(range(1, N_FOLDS + 1))
ax1.bar(fold_numbers, fold_val_accuracies, color='steelblue', alpha=0.7)
ax1.axhline(y=np.mean(fold_val_accuracies), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(fold_val_accuracies):.4f}')
ax1.set_xlabel('Fold', fontsize=12)
ax1.set_ylabel('Validation Accuracy', fontsize=12)
ax1.set_title('Validation Accuracy per Fold', fontsize=14, fontweight='bold')
ax1.set_xticks(fold_numbers)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

ax2.bar(fold_numbers, individual_fold_accuracies, color='forestgreen', alpha=0.7, label='Individual')
ax2.axhline(y=test_acc, color='red', linestyle='--', linewidth=2, label=f'Ensemble: {test_acc:.4f}')
ax2.axhline(y=np.mean(individual_fold_accuracies), color='orange', linestyle=':', linewidth=2, label=f'Mean: {np.mean(individual_fold_accuracies):.4f}')
ax2.set_xlabel('Fold', fontsize=12)
ax2.set_ylabel('Test Accuracy', fontsize=12)
ax2.set_title('Test Accuracy per Fold', fontsize=14, fontweight='bold')
ax2.set_xticks(fold_numbers)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('bilstm_cv_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print("Cross-validation summary plot saved as bilstm_cv_summary.png")

## 10. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
    cbar_kws={'label': 'Count'},
)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - Test Set (BiLSTM Ensemble)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('bilstm_cv_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("Confusion matrix saved as bilstm_cv_confusion_matrix.png")

## 11. Save Model and Artifacts

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
versioned_path = os.path.join(MODEL_SAVE_PATH, f'bilstm_cv_{timestamp}')
os.makedirs(versioned_path, exist_ok=True)

for i, model in enumerate(fold_models, 1):
    model_file = os.path.join(versioned_path, f'bilstm_fold_{i}.keras')
    model.save(model_file)
    print(f"Fold {i} model saved to {model_file}")

tokenizer_json = tokenizer.to_json()
with open(os.path.join(versioned_path, 'tokenizer.json'), 'w', encoding='utf-8') as f:
    f.write(tokenizer_json)

with open(os.path.join(versioned_path, 'label_encoder.json'), 'w', encoding='utf-8') as f:
    json.dump({
        'classes': label_encoder.classes_.tolist(),
    }, f, ensure_ascii=False, indent=2)

config = {
    'model_type': 'BiLSTM',
    'cross_validation': True,
    'n_folds': N_FOLDS,
    'vocab_size': vocab_size,
    'max_seq_len': MAX_SEQ_LEN,
    'embed_dim': EMBED_DIM,
    'lstm_units': LSTM_UNITS,
    'num_lstm_layers': NUM_LSTM_LAYERS,
    'num_classes': num_classes,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'random_seed': RANDOM_SEED,
    'underthesea_tokenizer': USE_UNDERTHESEA_TOKENIZER,
    'ensemble_test_accuracy': float(test_acc),
    'mean_cv_val_accuracy': float(np.mean(fold_val_accuracies)),
    'std_cv_val_accuracy': float(np.std(fold_val_accuracies)),
    'mean_cv_val_loss': float(np.mean(fold_val_losses)),
    'std_cv_val_loss': float(np.std(fold_val_losses)),
    'fold_val_accuracies': [float(acc) for acc in fold_val_accuracies],
    'fold_val_losses': [float(loss) for loss in fold_val_losses],
    'timestamp': timestamp,
}

with open(os.path.join(versioned_path, 'config.json'), 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print(f"\nAll artifacts saved to {versioned_path}")
print(f"  - {N_FOLDS} fold models (bilstm_fold_1.keras ... bilstm_fold_{N_FOLDS}.keras)")
print("  - tokenizer.json")
print("  - label_encoder.json")
print("  - config.json")

print("\n" + "="*80)
print("CROSS-VALIDATION COMPLETE")
print("="*80)
print(f"Mean Validation Accuracy: {np.mean(fold_val_accuracies):.4f} ± {np.std(fold_val_accuracies):.4f}")
print(f"Ensemble Test Accuracy: {test_acc:.4f}")
print("="*80)